# Seasonal Component Interactions Demo

This notebook demonstrates how to use the `x_cols_seasonal_interactions` parameter to create interactions between external features and seasonal patterns in time series forecasting.


## 1. Import Libraries and Create Demo Data


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from clairvoyants import Clairvoyant
from clairvoyants import ensemble
import matplotlib.pyplot as plt

# Create demo data with seasonal patterns and features
dates = pd.date_range('2020-01-01', '2022-12-31', freq='D')
n_days = len(dates)

# Base demand with trend
base_demand = 1000 + np.arange(n_days) * 0.5

# Seasonal patterns
weekly_pattern = 200 * np.sin(2 * np.pi * np.arange(n_days) / 7)  # Weekly seasonality
annual_pattern = 300 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)  # Annual seasonality

# Create features that should interact with seasonality
x_features = pd.DataFrame({'dt': dates})

# Marketing spend (should amplify seasonal patterns)
marketing_base = 1000
marketing_seasonal = 500 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + np.pi/2)
marketing_weekly = 200 * np.sin(2 * np.pi * np.arange(n_days) / 7)
marketing_trend = np.arange(n_days) * 1.0
marketing_noise = np.random.normal(0, 100, n_days)

x_features['marketing_spend'] = marketing_base + marketing_seasonal + marketing_weekly + marketing_trend + marketing_noise
x_features['marketing_spend'] = np.maximum(x_features['marketing_spend'], 0)

# Price (should dampen seasonal patterns)
price_base = 50
price_seasonal = 10 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + np.pi)
price_weekly = 5 * np.sin(2 * np.pi * np.arange(n_days) / 7 + np.pi)
price_trend = np.arange(n_days) * 0.05
price_noise = np.random.normal(0, 2, n_days)

x_features['price'] = price_base + price_seasonal + price_weekly + price_trend + price_noise

# Competitor activity (should affect seasonal strength)
competitor_base = 0.5
competitor_seasonal = 0.3 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)
competitor_noise = np.random.normal(0, 0.1, n_days)

x_features['competitor_activity'] = competitor_base + competitor_seasonal + competitor_noise
x_features['competitor_activity'] = np.clip(x_features['competitor_activity'], 0, 1)

# Economic indicator (should modulate overall seasonality)
economic_cycle = 0.2 * np.sin(2 * np.pi * np.arange(n_days) / (365.25 * 2))
economic_trend = np.arange(n_days) * 0.001
economic_noise = np.random.normal(0, 0.05, n_days)

x_features['economic_index'] = 1.0 + economic_cycle + economic_trend + economic_noise

# Create feature-seasonal interactions
marketing_interaction = (x_features['marketing_spend'] / 1000) * 0.3 * (weekly_pattern + annual_pattern)
price_interaction = -(x_features['price'] / 50) * 0.2 * (weekly_pattern + annual_pattern)
competitor_interaction = -x_features['competitor_activity'] * 0.4 * (weekly_pattern + annual_pattern)
economic_interaction = (x_features['economic_index'] - 1) * 0.5 * (weekly_pattern + annual_pattern)

# Combine all effects
feature_interactions = marketing_interaction + price_interaction + competitor_interaction + economic_interaction

# Add noise
noise = np.random.normal(0, 50, n_days)

# Calculate final demand
actual = base_demand + weekly_pattern + annual_pattern + feature_interactions + noise
actual = np.maximum(actual, 0)

# Create main DataFrame
df = pd.DataFrame({
    'dt': dates,
    'actual': actual
})

print(f"Generated {len(df)} days of data")
print(f"Date range: {df['dt'].min()} to {df['dt'].max()}")
print(f"Actual values: {df['actual'].min():.1f} to {df['actual'].max():.1f}")
